In [ ]:
# Prachi
# 8025320072
# Assignment 6

In [ ]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 7.9 MB/s eta 0:00:00


In [ ]:
# Q1 Word Count

from collections import defaultdict

data = ["hadoop is fast", "hadoop is scalable"]

mapped = []
for line in data:
    for word in line.split():
        mapped.append((word, 1))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

result = {k: sum(v) for k, v in shuffled.items()}

print(result)

{'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


In [ ]:
%%writefile q1.py
from mrjob.job import MRJob

class WordCount(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Writing q1.py


In [ ]:
%%writefile input1.txt
hadoop is fast
hadoop is scalable

Writing input1.txt


In [ ]:
!python q1.py input1.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q1.root.20260425.183801.492364
Running step 1 of 1...
job output is in /tmp/q1.root.20260425.183801.492364/output
Streaming final output from /tmp/q1.root.20260425.183801.492364/output...
"is"	2
"scalable"	1
"fast"	1
"hadoop"	2
Removing temp directory /tmp/q1.root.20260425.183801.492364...


In [ ]:
# Q2 Character Count

from collections import defaultdict

data = "big data"

mapped = [(ch, 1) for ch in data if ch != " "]

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

result = {k: sum(v) for k, v in shuffled.items()}
print(result)

{'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


In [ ]:
%%writefile q2.py
from mrjob.job import MRJob

class CharCount(MRJob):

    def mapper(self, _, line):
        for ch in line:
            if ch != " ":
                yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Writing q2.py


In [ ]:
%%writefile input2.txt
big data

Writing input2.txt


In [ ]:
!python q2.py input2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q2.root.20260425.183914.838617
Running step 1 of 1...
job output is in /tmp/q2.root.20260425.183914.838617/output
Streaming final output from /tmp/q2.root.20260425.183914.838617/output...
"b"	1
"d"	1
"g"	1
"i"	1
"t"	1
"a"	2
Removing temp directory /tmp/q2.root.20260425.183914.838617...


In [ ]:
# Q3 Average Word Length

from collections import defaultdict

data = ["big data"]

mapped = []
for line in data:
    for word in line.split():
        mapped.append((word, len(word)))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

result = {k: sum(v)/len(v) for k, v in shuffled.items()}
print(result)

{'big': 3.0, 'data': 4.0}


In [ ]:
%%writefile q3.py
from mrjob.job import MRJob

class AvgWordLength(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield word, len(word)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    AvgWordLength.run()

Writing q3.py


In [ ]:
%%writefile input3.txt
big data

Writing input3.txt


In [ ]:
!python q3.py input3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q3.root.20260425.184014.055841
Running step 1 of 1...
job output is in /tmp/q3.root.20260425.184014.055841/output
Streaming final output from /tmp/q3.root.20260425.184014.055841/output...
"data"	4.0
"big"	3.0
Removing temp directory /tmp/q3.root.20260425.184014.055841...


In [ ]:
# Q4 Global Average Word Length

data = ["hadoop mapreduce spark"]

words = []
for line in data:
    words.extend(line.split())

avg = sum(len(w) for w in words) / len(words)

print(avg)

6.666666666666667


In [ ]:
%%writefile q4.py
from mrjob.job import MRJob

class GlobalAvg(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield "avg", (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        count = 0
        for l, c in values:
            total_len += l
            count += c
        yield key, total_len / count

if __name__ == "__main__":
    GlobalAvg.run()

Writing q4.py


In [ ]:
%%writefile input4.txt
hadoop mapreduce spark

Writing input4.txt


In [ ]:
!python q4.py input4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q4.root.20260425.184110.676891
Running step 1 of 1...
job output is in /tmp/q4.root.20260425.184110.676891/output
Streaming final output from /tmp/q4.root.20260425.184110.676891/output...
"avg"	6.666666666666667
Removing temp directory /tmp/q4.root.20260425.184110.676891...


In [ ]:
# Q5

from google.colab import files
uploaded = files.upload()

file = open("shakespeare.txt", "r")
data = file.readlines()

print(data[:5])  # preview

from collections import defaultdict

mapped = []

for line in data:
    for word in line.strip().split():
        mapped.append((word.lower(), 1))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

word_count = {k: sum(v) for k, v in shuffled.items()}

print(list(word_count.items())[:10])

top5 = sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 words:", top5)

mapped = []

for line in data:
    for ch in line:
        if ch != " " and ch != "\n":
            mapped.append((ch, 1))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

char_count = {k: sum(v) for k, v in shuffled.items()}

print(list(char_count.items())[:10])

mapped = []

for line in data:
    for word in line.split():
        mapped.append((word.lower(), len(word)))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

avg_word_len = {k: sum(v)/len(v) for k, v in shuffled.items()}

print(list(avg_word_len.items())[:10])

words = []

for line in data:
    words.extend(line.split())

avg = sum(len(w) for w in words) / len(words)

print("Global Average Length:", avg)

Saving shakespeare.txt to shakespeare (1).txt
['\ufeffThe Project Gutenberg EBook of The Complete Works of William Shakespeare, by\n', 'William Shakespeare\n', '\n', 'This eBook is for the use of anyone anywhere at no cost and with\n', 'almost no restrictions whatsoever.  You may copy it, give it away or\n']
[('\ufeffthe', 1), ('project', 320), ('gutenberg', 250), ('ebook', 13), ('of', 18126), ('the', 27729), ('complete', 243), ('works', 268), ('william', 311), ('shakespeare,', 2)]
Top 5 words: [('the', 27729), ('and', 26099), ('i', 19540), ('to', 18762), ('of', 18126)]
[('\ufeff', 1), ('T', 39878), ('h', 218875), ('e', 406157), ('P', 12038), ('r', 209907), ('o', 282560), ('j', 2788), ('c', 67194), ('t', 291243)]
[('\ufeffthe', 4.0), ('project', 7.0), ('gutenberg', 9.0), ('ebook', 5.0), ('of', 2.0), ('the', 3.0), ('complete', 8.0), ('works', 5.0), ('william', 7.0), ('shakespeare,', 12.0)]
Global Average Length: 4.484685214825106


In [ ]:
%%writefile q5_wordcount.py
from mrjob.job import MRJob

class WordCount(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield word.lower(), 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

In [ ]:
# Q6 Average Marks Per Student

from collections import defaultdict

data = ["A 80", "B 70", "A 90", "B 60", "A 100"]

mapped = []
for line in data:
    name, marks = line.split()
    mapped.append((name, int(marks)))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

result = {k: sum(v)/len(v) for k, v in shuffled.items()}
print(result)

{'A': 90.0, 'B': 65.0}


In [ ]:
%%writefile q6.py
from mrjob.job import MRJob

class AvgMarks(MRJob):

    def mapper(self, _, line):
        name, marks = line.split()
        yield name, int(marks)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    AvgMarks.run()

Writing q6.py


In [ ]:
%%writefile input6.txt
A 80
B 70
A 90
B 60
A 100

Writing input6.txt


In [ ]:
!python q6.py input6.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q6.root.20260425.184228.893381
Running step 1 of 1...
job output is in /tmp/q6.root.20260425.184228.893381/output
Streaming final output from /tmp/q6.root.20260425.184228.893381/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/q6.root.20260425.184228.893381...


In [ ]:
# Q7 Average Salary + Highest Dept

from collections import defaultdict

data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

mapped = []
for line in data:
    dept, sal = line.split()
    mapped.append((dept, int(sal)))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

avg_salary = {k: sum(v)/len(v) for k, v in shuffled.items()}

highest = max(avg_salary, key=avg_salary.get)

print(avg_salary)
print("Highest:", highest)

{'HR': 55000.0, 'IT': 75000.0}
Highest: IT


In [ ]:
%%writefile q7.py
from mrjob.job import MRJob

class Salary(MRJob):

    def mapper(self, _, line):
        dept, sal = line.split()
        yield dept, int(sal)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    Salary.run()

Writing q7.py


In [ ]:
%%writefile input7.txt
HR 50000
IT 70000
HR 60000
IT 80000

Writing input7.txt


In [ ]:
!python q7.py input7.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q7.root.20260425.184420.927345
Running step 1 of 1...
job output is in /tmp/q7.root.20260425.184420.927345/output
Streaming final output from /tmp/q7.root.20260425.184420.927345/output...
"IT"	75000.0
"HR"	55000.0
Removing temp directory /tmp/q7.root.20260425.184420.927345...


In [ ]:
# Q8 Average Temperature Per City

from collections import defaultdict

data = ["New York,38", "London,29", "Tokyo,35", "New York,32"]

mapped = []
for line in data:
    city, temp = line.split(",")
    mapped.append((city, int(temp)))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

result = {k: sum(v)/len(v) for k, v in shuffled.items()}
print(result)

{'New York': 35.0, 'London': 29.0, 'Tokyo': 35.0}


In [ ]:
%%writefile q8.py
from mrjob.job import MRJob

class Temp(MRJob):

    def mapper(self, _, line):
        city, temp = line.split(",")
        yield city, int(temp)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    Temp.run()

Writing q8.py


In [ ]:
%%writefile input8.txt
New York,38
London,29
Tokyo,35
New York,32

Writing input8.txt


In [ ]:
!python q8.py input8.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q8.root.20260425.184525.338329
Running step 1 of 1...
job output is in /tmp/q8.root.20260425.184525.338329/output
Streaming final output from /tmp/q8.root.20260425.184525.338329/output...
"New York"	35.0
"Tokyo"	35.0
"London"	29.0
Removing temp directory /tmp/q8.root.20260425.184525.338329...


In [1]:
# Q9

from google.colab import files
uploaded = files.upload()

import pandas as pd

df = pd.read_csv("/content/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv")

df.head()

df.columns

df = df[['Country', 'AverageTemperature']]
df = df.dropna()
result = df.groupby('Country')['AverageTemperature'].mean()

print(result.head(10))
top = result.sort_values(ascending=False).head(10)

print("Top hottest countries:")
print(top)

from collections import defaultdict

mapped = []

for _, row in df.iterrows():
    mapped.append((row['Country'], row['AverageTemperature']))

shuffled = defaultdict(list)
for k, v in mapped:
    shuffled[k].append(v)

reduced = {k: sum(v)/len(v) for k, v in shuffled.items()}

# print sample
for i, (k, v) in enumerate(reduced.items()):
    if i == 10:
        break
    print(k, v)


Saving GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv to GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity (1).csv
Country
Afghanistan    14.342919
Angola         23.693046
Australia      15.190055
Bangladesh     25.490568
Brazil         22.847555
Burma          26.735193
Canada          5.109462
Chile           5.692277
China          11.793666
Colombia       20.892248
Name: AverageTemperature, dtype: float64
Top hottest countries:
Country
Sudan           29.081291
Vietnam         27.193984
Thailand        27.164733
Somalia         27.151964
Burma           26.735193
Indonesia       26.659057
Singapore       26.523103
Philippines     26.448334
Saudi Arabia    26.427309
Nigeria         26.361323
Name: AverageTemperature, dtype: float64
Côte D'Ivoire 26.16373719752392
Ethiopia 17.52507266229899
India 25.80930923845554
Syria 17.37058733360226
Egypt 20.900406225165565
Turkey 13.790997727026735
Iraq 22.61434568965517
Thailand 27.164733303650937
Brazil 22.84755523519235

In [2]:
df.to_csv("temp_data.csv", index=False)

In [3]:
%%writefile q9.py
from mrjob.job import MRJob
import csv

class Temp(MRJob):

    def mapper(self, _, line):
        try:
            row = list(csv.reader([line]))[0]
            if row[0] == "Country":
                return
            country = row[0]
            temp = float(row[1])
            yield country, temp
        except:
            pass

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    Temp.run()

Writing q9.py


In [4]:
!python q9.py temp_data.csv

Traceback (most recent call last):
  File "/content/q9.py", line 1, in <module>
    from mrjob.job import MRJob
ModuleNotFoundError: No module named 'mrjob'
